# 4. Decoder-Only Transformer

Decoder-only transformers (like GPT) use masked self-attention and feed-forward networks.
This architecture powers modern language models!


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


## Key Components

1. **Masked Multi-Head Attention** - Can only attend to previous positions
2. **Feed-Forward Network** - Two linear layers with activation
3. **Layer Normalization** - Normalizes activations
4. **Residual Connections** - Adds input to output (x + f(x))


## Creating mask
- It used on hiding the future information to avoid model cheating on 'looking' the future
- Ensuring the model the predicting what are the next token instead of 'copying the answer'

In [24]:
# defining sequence_length
seq_len = 5

# creating input 
x = torch.ones(seq_len, seq_len)

# Masked attention: prevent attending to future positions
# Causal mask or known as look-ahead mask: prevent model from 'looking' into future
#   In training, the mask used to prevent the model to 'see' the answer without predicting.
#   Ensuring the model only attends word from specific range to specific range

# Create causal mask (higher triangular) by filling with ones
#   Example: x - (5,5) tensor object, the matrix shape square are split into 2 part (or 2 triangles)
#               this initialize the higher part of triangle (higher triangular - triu()) filled in 1s
#   Diagonal = 1 used to shift the diagonal up by = 1 as offset from the 'split' of diagonal
mask = torch.triu(x, diagonal=1)
print(f"Mask shape: {mask.shape}")
print(mask)
print()

# now, defining the if the mask == 1(true), then it becomes -inf
mask = mask.masked_fill(mask == 1, float('-inf'))
# if the mask == 0(false), its sets it value as 0.0
mask = mask.masked_fill(mask == 0, 0.0)
print("Masking..")
print("Causal mask, changing the value (0 -> allowed, -inf -> masked):")
print(mask)

print("\nEach position can only attend to itself and previous positions!")
print("Limit to: allowed position = current position - 1")


Mask shape: torch.Size([5, 5])
tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])

Masking..
Causal mask, changing the value (0 -> allowed, -inf -> masked):
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

Each position can only attend to itself and previous positions!
Limit to: allowed position = current position - 1


## (1) Masked Multi-Head Attention
- Enables tokens to talk one another and update its information
    - Process where which tokens talk which other tokens
- Same as multi-head attention, but with causal masking applied to attention scores.
    - Multi-head attention + masking (causal mask)
- From x (input) -> x's learnable representation
    - executing the forward process from x -> x's input representation 
    - through x 
    -          -> Q(x), K(x), V(x) 
    -           -> reshape in multi-heads 
    -           -> calculate attention_score QK^T/ sqrt(d_k)
    -           -> apply masking
    -           -> softmax(attention_score)
    -           -> multiply with V (to perform complete attention_score QK^T/sqrt(d_k))
    -           -> reshape back multi-head into one as attention_output
    -           -> entering attention_output into output linear projection


In [ ]:
class MaskedMultiHeadAttention(nn.Module):
    """Masked multi-head attention for decoder"""
    
    def __init__(self, d_model, num_heads):
        # inheriting the pytorch NN libraries
        super().__init__()
        
        # ensuring the dimension_model (d_model) is divisible by number_heads (num_heads)
        #   if true, the runtime will pass and run the execution
        #   else (false), the runtime will halt and throws AssertionError Exception
        # it is to ensure the divisible d_model % num_heads == 0 for splitting across multi-heads 
        assert d_model % num_heads == 0
        
        # d_model = dimension of model (how much numbers in each token)
        self.d_model = d_model
        # num_heads = how much heads in multi-head attentions
        self.num_heads = num_heads

        # calculating dimension_key (d_k)
        #   done by floor division (// operation) by 
        #       ignoring the float value and round it off the integer value 
        self.d_k = d_model // num_heads # <-- ensured only d_model %% num_heads == 0
        
        # initializing the weights for Q(Query), K(Key), V(Value) and O(Output)
        #   in linear projection
        #   in defining the features = d_model (for representation)
        # defining 'bias=False' 
        #   - stick the formula for calculating attention score (not including bias)
        #   - reducing unecessary trainable parameters
        #   - biases are needed but not in here 
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        # unlike Q, K and V - initializing the weights for O(Output)
        #   in linear projection by defining the feature = d_model
        #   withouth 'bias=False', therefore bias is enabled
        self.W_o = nn.Linear(d_model, d_model)

    # executing the forward process from x -> x's input representation 
    #   through x 
    #           -> Q(x), K(x), V(x) 
    #           -> reshape in multi-heads 
    #           -> calculate attention_score QK^T/ sqrt(d_k)
    #           -> apply masking
    #           -> softmax(attention_score)
    #           -> multiply with V (to perform complete attention_score QK^T/sqrt(d_k))
    #           -> reshape back multi-head into one as attention_output
    #           -> entering attention_output into output linear projection
    def forward(self, x, mask=None):
        """
        Args:
            x: [batch_size, seq_len, d_model]
            mask: Optional causal mask [seq_len, seq_len]
        """
        # taking batch_size, seq_len and d_model taken from x.shape
        batch_size, seq_len, d_model = x.shape
        
        # inserting the x input into linear weights for Q, K and V 
        #   to produce Q, K and V represntation of X's input
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Reshape for multi-head (splicing one overall linear projection into multiple multi-heads)
        #   by referencing teh original shape (batch_size, seq_len, d_model)
        #       into multi-head (batch_size, seq_len, num_heads, d_k)
        #   then, transpose by swapping dimension dim1 and dim2 becomes
        #        (batch_seize, num_heads, seq_len, d_k) <-- requirement for matmul later on
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        #   Calculate attention_score (QK^T / sqrt(d_k)) (1st and 2nd step)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Show attention_score dimension
        # print(f"Attention score shape: {scores.shape}")
        # print()

        # Show one part of attention_scores before masking
        #   Remember: array dimension can be represent in 3-dimension
        #           the print() can be represented in 3 innermost dimension
        # print("Attention score before masking (values inside tensor object as tokens):")
        # print("(num_heads, seq_len, seq_len)")
        # print(f"Attention_score[0,0,0] = {scores[0,0,0]}")
        # print(f"Attention_score[0,0,1] = {scores[0,0,1]}")
        # print()

        # Apply causal mask if provided
        #   ensuring the model actually predicting instead of 'copying' the future tokens
        # causal mask properties defined in main execution
        # copy the masking from 'mask' provided into attention_scores
        if mask is not None:
            scores = scores.masked_fill(mask == float('-inf'), float('-inf'))

        # Show one part of same attention_scores after masking
        # print("Attention score after applying causal_masking (values inside tensor object as tokens):")
        # print("(num_heads, seq_len, seq_len)")
        # print(f"Attention_score[0,0,0] = {scores[0,0,0]}")
        # print(f"Attention_score[0,0,1] = {scores[0,0,1]}")
        # print()
        
        # Calculate attention weight (3rd step)
        #   by calculating ranging through softmax in innermost layer (0 - 1)
        attention_weights = F.softmax(scores, dim=-1)
        # print(f"Attention weight shape: {attention_weights.shape}")
        # print()
        # print("Masking also applied here after the softmax() (sets to zero(0))")
        # print(f"Attention_weight[0,0,0] = {attention_weights[0,0,0]}")
        # print(f"Attention_weight[0,0,1] = {attention_weights[0,0,1]}")
        # print()

        # Calculate the output (4th step)
        attention_output = torch.matmul(attention_weights, V)
        
        # Concatenate heads
        # Combine the multi-heads to single Q, K and V vectors
        attention_output = attention_output.transpose(1, 2).contiguous()
        # Reshape back into original (batch_size, seq_len, d_model)
        attention_output = attention_output.view(batch_size, seq_len, d_model)
        
        # Calculate the output the projecting the attention_output as input
        #   in weight output
        # The usage on entering attention_output in linear projection 
        #   to combine all calculations inside linear projection
        #   from multi head attentions instead of just concatenating
        # Also, get the x's input representation output from multi-head attention
        #   and combine back together
        output = self.W_o(attention_output)
        return output

# Test masked attention
batch_size = 2
seq_len = 10
num_heads = 8
d_model = 64

# Create causal mask
# Create causal mask that similar dimension as attention_scores (seq_len, seq_len)
# in higher higher triangle diagonal with offset = 1 (diagonal=1)
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
# set the mask properties 1 -> -inf
causal_mask = causal_mask.masked_fill(causal_mask == 1, float('-inf'))

# Creating masked multi head attention with x input
mha = MaskedMultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)

print(f"Input shape: {x.shape}")
print()
print(f"Weight for Q, K and V (bias=False): {mha.W_q}")
print(f"Weight for O (bias=True): {mha.W_o}")
print()
output = mha(x, mask=causal_mask)
print(f"Output shape: {output.shape}")
print("Masked multi-head attention with causal masking!")


Input shape: torch.Size([2, 10, 64])

Weight for Q, K and V (bias=False): Linear(in_features=64, out_features=64, bias=False)
Weight for O (bias=True): Linear(in_features=64, out_features=64, bias=True)

Output shape: torch.Size([2, 10, 64])
Masked multi-head attention with causal masking!


## (2) Applying FeedFoward Network

- Each updated token will be processed deeply
    - Process where the information gathered from multi-head attentions into use based on context given 
- This is where linear projection with activation function will be done
    - Typically, dimension of feedforward (d_ff) will be 4x bigger than d_model
    - Later, it will scaled down back to d_model
- From x's learnable representation -> x's actual information
    - executing the forward process from x's learnable representation -> x's actual information needed
    -   through x's learnable representation 
    -           -> transform in linear1 projection (similar as forward() gradient in NN - 1st layer)
    -           -> through activation function (ReLU)
    -           -> transform in lenear2 projectin (similar as forward() gradient in NN - 2nd)

In [ ]:
# feed forward class
class FeedForward(nn.Module):
    """Feed-forward network"""
    
    # inititializing..
    def __init__(self, d_model, d_ff):
        # inheritng from PyTorch libraries
        super().__init__()

        # initializing non-square matrix linear projectors..
        # first linear projection with dimension (m,n) (dimension_model, dimension_feedForward)
        # expands the dimension from d_model -> d_ff
        self.linear1 = nn.Linear(d_model, d_ff)

        # second linear projection with dimension (n,m) (dimension_feedForward, dimension_model)
        # for multiplication
        # will return back into original dimension d_ff -> d_model
        self.linear2 = nn.Linear(d_ff, d_model)

        # initializing activation function by using ReLU
        self.relu = nn.ReLU()
        
    # executing the forward process from x's learnable representation -> x's actual information needed
    #   through x's learnable representation 
    #           -> transform in linear1 projection (similar as forward() gradient in NN - 1st layer)
    #           -> through activation function (ReLU)
    #           -> transform in lenear2 projectin (similar as forward() gradient in NN - 2nd)
    def forward(self, x):
        # Expand: d_model -> d_ff, then contract: d_ff -> d_model
        # execute self.linear1(x) -> return projected x from linear fx 1 ->
        #           activate function ReLU(projected x from linear fx 1) -> activated function ->
        #           self.linear2(activated function from linear projected x linear1)                      
        # Original code: return self.linear2(self.relu(self.linear1(x)))
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)

        return x

# Test feed-forward
# difference between dimension_model (d_model) vs dimension_feedForward (d_ff)
# d_model - Transformer's main hidden size
#         - width of vectors flowing through the model (token embeddings, attention inputs/outputs, residual streams)
# d_ff - hidden size of feed-forward network in each Transformer block (inner expansion)
#      - usually larger than d_model becasue it temporarily expands representation
#         - applies non-linearity (activation function)
#         - later it projects back down (d_model)
batch_size = 2
seq_len = 10
d_model = 64
# d_ff typically 4x d_model
# d_ff = 256
d_ff = 4 * d_model

# Create feedforward network..
ff = FeedForward(d_model, d_ff)

# Create input x tensor..
x = torch.randn(batch_size, seq_len, d_model)

# Executing feedForward by passing x as input argument
output = ff(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print()
print(f"Expanded to {d_ff} dimensions, then back to {d_model}")


Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])

Expanded to 256 dimensions, then back to 64


## Transformer Decoder Block

- Combines masked attention and feed-forward with addition: 
    - layer normalization
        - used for normalization to ensure the training is stable within sublayers
    - dropout
        - used for regularization process happen
            - Zeroes out some activations
            - Doesn't too much rely on specific neurons or paths
            - Reduce overfitting, improves generalization and makes model more robust
        - used in training
            - after embeddings 
            - after multi-head attention outputs
            - after feed-forward outputs
    - residual connections.
        - the original input information to ensure the original information still can be referred even after all the heavy processing in (1) multi-head attention and (2) feed-forward network
- executing the forward process from x -> next hidden states/ final layer norm (process in output head)
-   through x
-           -> (1) multi-head attention with mask + dropout + residual + normalization layer
-              -> get attention_output from (1) multi-head attention with mask
-              -> through dropout() for generalizability and avoid overfitting
-              -> adding back in x original information (residual) for reference
-              -> enter through normalization layer to ensure training stable
-           -> (2) feed-forward network + dropout + residual + normalization layer
-              -> get feed_forward from (2) feed_forward network
-              -> through dropout() for generability and avoid overfitting
-              -> adding back in x original infomraiton (residual) for reference
-              -> enter through normalization layer to ensure training stable

In [ ]:
# transfomerDecoderBlock to utilize the classes above
class TransformerDecoderBlock(nn.Module):
    """Single decoder block"""
    
    # initializing..
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        # inheriting the super class from PyTorch NN libraries
        super().__init__()

        # Utilize (1) masked multi head attention
        #   This is where linear projection for Q, K and V with passing argument for 
        #   (d_model, num_heads)
        #   This is where the tokens, matrices to interact one another to gain more information about the context
        self.attention = MaskedMultiHeadAttention(d_model, num_heads)

        # (2) Utilize feed forward network
        #   This is where linear projection with activation function with passing argument for
        #   (d_model, d_ff)
        #   This is where the each specific tokens, matrices to be processed in feed forward path in specific  context
        #       that requires temporary larger dimensions FF (d_model * 4)
        self.feed_forward = FeedForward(d_model, d_ff)

        # Utilize layer norm
        #   This is where normalization across feature dimensions and information executing processes each layer
        #   Ensure the training phase is stable within sublayers in hidden layers
        #   Since it have 2 layers, so utilize norm1 and norm2
        #       (1) multi-head attention with masking and 
        #       (2) forward-feed network 
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Utilize dropout
        #   This is where regularization happens to zeroes out some activations
        #   so model doesn't too much rely on specific neurons or paths
        #   This is used to reduce overfitting, improves generalization and makes model more robust
        #   This only used on training
        self.dropout = nn.Dropout(dropout)
    
    # executing the forward process from x -> next hidden states/ final layer norm (process in output head)
    #   through x
    #           -> (1) multi-head attention with mask + dropout + residual + normalization layer
    #              -> get attention_output from (1) multi-head attention with mask
    #              -> through dropout() for generalizability and avoid overfitting
    #              -> adding back in x original information (residual) for reference
    #              -> enter through normalization layer to ensure training stable
    #           -> (2) feed-forward network + dropout + residual + normalization layer
    #              -> get feed_forward from (2) feed_forward network
    #              -> through dropout() for generability and avoid overfitting
    #              -> adding back in x original infomraiton (residual) for reference
    #              -> enter through normalization layer to ensure training stable
    def forward(self, x, mask=None):
        # Self-attention with dropout, residual and norm
        # excecute multi head attentions with residual and normalization
        # Attention - used for mix information between tokens and words
        attn_output = self.attention(x, mask)
        # from attention_output -> regularize through dropout -> adds with x input -> normalized in first layer
        # original code: x = self.norm1(x + self.dropout(attn_output))
        # attention output -> regularize through dropout
        dropped_out = self.dropout(attn_output)
        # regularize through dropout -> adds with x input (as residual)
        # adds back the original information of x input after dropped_out (ensuring it remembers the original information)
        residual = x + dropped_out
        # adds with x input (as residual) -> normalized in first layer
        x = self.norm1(residual)

        # Feed-forward with dropout, residual and norm
        # Executes the feed forward with residual and normalization
        # feed_forward - used for transform information within each token
        ff_output = self.feed_forward(x)
        # from ff_output -> regularize through dropout -> adds with x input -> normalized in second layer
        # original code: x = self.norm2(x + self.dropout(ff_output))
        # ff_output -> regularize through droput
        dropped_out = self.dropout(ff_output)
        # regularize through droput -> adds with x input (as residual)
        # adds back the original information of x input after dropped_out (ensuring it remembers the original information)
        residual = x + dropped_out
        # adds with x input (as residual) -> normalized in second layer
        x = self.norm2(residual)
        
        return x

# Test decoder block
batch_size = 2
seq_len = 10
# ensuring d_model % num_heads == 0
num_heads = 8
d_model = 64
# typically d_ff = d_model * 4
d_ff = 256

# Defining causal_mask
#   with dimension (seq_len, seq_len) filling upper triangle diagonal in shifting to = 1 as ones
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
# Later update the masked_fill the value = '1' into '-inf'
causal_mask = causal_mask.masked_fill(causal_mask == 1, float('-inf'))

# Defining transformerDecoderBlock object with dimension (d_model, num_heads, d_ff)
block = TransformerDecoderBlock(d_model, num_heads, d_ff)

# Defining random x input using dimension (batch_size, seq_len, d_model)
x = torch.randn(batch_size, seq_len, d_model)

print(f"Input shape: {x.shape}")

# executing the transformer
# basically it calls the forward() inside TransformerDecoderBlock class
#   or calls block.forward(x, mask=causal_mask)
output = block(x, mask=causal_mask) 

print(f"Output shape: {output.shape}")
print("Decoder block with masked attention, feed-forward, dropout, residuals and normalized!")


Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
Decoder block with masked attention, feed-forward, dropout, residuals and normalized!


## Complete Decoder-Only Transformer

- Stack multiple decoder blocks to create a full transformer model.
    - Embedding part (token + position)
    - Dropout
    - Transfomer block
        - Multi head attention
        - Dropout
        - Feed Forward
        - Dropout
        - Residual connection
        - Normalization layer
    - Output layer
        - linear projection
        - normalization layer

TO-DO:
- comparing backpropagation() in Transformer
    - loss function
    - backward()
    - optimizer

In [ ]:
# class definition on combining all everything as one complete Decoder only Transformer
class DecoderOnlyTransformer(nn.Module):
    """Decoder-only transformer (GPT-style)"""
    
    # initializing accepts args:
    #   - vocab size (token_embedding)
    #   - d_model (for everything inside the models)
    #   - num_heads (for multi heads attentions)
    #   - num_layers (defining number of layers in decoder transformer block)
    #   - d_dff (for feedforward network in forward() - typically d_model * 4)
    #   - max_seq_len (position_embedding)
    #   - dropout - dropout rate to regularization
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # EMBEDDING PART
        # Token and positional embeddings
        # Token embedding: represents what are the tokens
        #                       - captures the meaning of the word
        #                       - contains semantic identity
        # Passes arguments
        #   - vocab_size: how many different unique tokens exists/entries
        #   - d_model: how many numbers represent each token (length of tokens)
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # position_embedding: represents the token in sequence
        #                       - captures the index inside sentence
        #                       - contains order information
        # Passes arguments
        #   - max_seq_len: how many positions model to support
        #   - d_model: how many nymberrs represent each position
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        
        # TRANSFORMER DECODER BLOCK
        # Stack of decoder blocks
        # initialize Transformer Decoder Block as stacks (bunch decoder blocks)
        self.blocks = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # OUTPUT LAYER
        # Output projection
        # By creating linear projection and final normalization
        #   transform back into dimension (d_model, vocab_size, bias)
        #   similar to x input
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Creating dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        Args:
            x: Token indices [batch_size, seq_len]
        Returns:
            logits: [batch_size, seq_len, vocab_size]
        """

        # define batch_size and seq_len from x's dimension
        batch_size, seq_len = x.shape
        
        # Create causal mask
        #   by filling ones in higher triangle and set offset=1 upwards
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1)
        # Change value == 1 -> '-inf'
        mask = mask.masked_fill(mask == 1, float('-inf'))
        
        # Token embeddings
        #   store the x input into token embedder
        token_emb = self.token_embedding(x)  # [batch_size, seq_len, d_model]
        
        # Positional embeddings
        #   Create index number using arange() based on
        #   seq_len = number limit
        #   device = store as same as x's memory location (system RAM or VRAM)
        #   unsqueeze() = adds dimension at outermost dimension (dim=0)
        #                   dim (seq_len) -> (1, seq_len)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        # applies the position_embedding() initialize earlier by passing 
        #   positions argument
        # Therefore, pos_emb will have 3 dimensions (1, seq_len, d_model)
        #   1 - 1 (batch_dimension)
        #      '1' in batch_size dimension indicates the broadcast of 'values' 
        #           to (batch_size). For example, if batch_size = 3, the token_emb
        #           refers the same '1' as pos_emb references
        #   seq_len - seq_len (embedding position's per token)
        #   d_model - how many numbers represented per token (embedding size)
        pos_emb = self.position_embedding(positions)  # [1, seq_len, d_model]
        
        # Combine embeddings
        # Original code: x = self.dropout(token_emb + pos_emb)
        # Combine token_emb and pos_emb through sum operation
        #   to represent the word embedding of x's input (token_emb + pos_emb) signal
        #   before passing to transformer block
        x = token_emb + pos_emb
        # use dropout() after merging token_emb + pos_emb
        #   to regularieze full input representation
        #       - reduce overfitting to avoid relying on specific exact embedding features
        #       - encourage robustness and generalizability
        #       - avoid depending on positional cues or token identify cues alone
        x = self.dropout(x)

        # Pass through decoder blocks
        for block in self.blocks:
            # calls TransformerDecoderBlock class above, to perform forward()
            x = block(x, mask=mask)
        
        # Final layer norm and projection
        #   performing normalization overall representation at final layer (clean output)
        #       rather than intermediate computations (safe processing) in sublayer in hidden layers
        #      for preparation to output layer
        #      ensuring values (hidden features) are well-scaled and consistent
        x = self.ln_f(x)

        # turns the x's clean output from self.ln_f() into
        #   vocabulary scores to predict the next token
        # Uses to translate hidden features into token prediction scores
        #    x's clean output -> x's vocabulary scores (logits = score for each possible next token)
        logits = self.head(x)
        
        return logits

# Test complete transformer
vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 4
d_ff = 2048
max_seq_len = 1024

model = DecoderOnlyTransformer(
    vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len
)

# Example input: token indices
batch_size = 2
seq_len = 20

# Create x's input with random int from 0 <= x < vocab-size
#   with dimension of (batch_size, seq_len)
x = torch.randint(0, vocab_size, (batch_size, seq_len))

logits = model(x)

print(f"Input shape: {x.shape}")
print(f"Output logits shape: {logits.shape}")
print()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Model parameters breakdown:")
print("Consist of:")
print("     Embedding (token + position)")
print("     Transfomer Block (based on number of Transformer decoder blocks):")
print("         Attention weight (Q - Query, K - Key, V - Value, O - output)")
print("         Attention bias (O - output - as output only have Bias = 'True')")
print("         Normalizer layers (weight and bias)")
print("     Final projection layer")
print("     Final output head weight")
print()
for name, p in model.named_parameters():
    print(name, p.shape, p.numel())
print("\nDecoder-only transformer complete!")


Input shape: torch.Size([2, 20])
Output logits shape: torch.Size([2, 20, 1000])

Model parameters: 1,080,576
Model trainable parameters: 1,080,576
Model parameters breakdown:
Consist of:
     Embedding (token + position)
     Transfomer Block (based on number of Transformer decoder blocks):
         Attention weight (Q - Query, K - Key, V - Value, O - output)
         Attention bias
         Normalizer layers (weight and bias)
     Final projection layer
     Final output head weight

token_embedding.weight torch.Size([1000, 128]) 128000
position_embedding.weight torch.Size([256, 128]) 32768
blocks.0.attention.W_q.weight torch.Size([128, 128]) 16384
blocks.0.attention.W_k.weight torch.Size([128, 128]) 16384
blocks.0.attention.W_v.weight torch.Size([128, 128]) 16384
blocks.0.attention.W_o.weight torch.Size([128, 128]) 16384
blocks.0.attention.W_o.bias torch.Size([128]) 128
blocks.0.feed_forward.linear1.weight torch.Size([512, 128]) 65536
blocks.0.feed_forward.linear1.bias torch.Size([51